---
title: Another Kind of Character
---

In [1]:
import polars as pl
import hvplot.polars
path_data = '../../../../data/'

from urllib.request import urlopen 
import re
def read_url(url): 
    return re.sub('\\s+', ' ', urlopen(url).read().decode())

In [2]:
# Read two books, fast (again)!

huck_finn_url = 'https://www.inferentialthinking.com/data/huck_finn.txt'
huck_finn_text = read_url(huck_finn_url)
huck_finn_chapters = huck_finn_text.split('CHAPTER ')[44:]

little_women_url = 'https://www.inferentialthinking.com/data/little_women.txt'
little_women_text = read_url(little_women_url)
little_women_chapters = little_women_text.split('CHAPTER ')[1:]

In some situations, the relationships between quantities allow us to make predictions. This text will explore how to make accurate predictions based on incomplete information and develop methods for combining multiple sources of uncertain information to make decisions.

As an example of visualizing information derived from multiple sources, let us first use the computer to get some information that would be tedious to acquire by hand. In the context of novels, the word "character" has a second meaning: a printed symbol such as a letter or number or punctuation symbol. Here, we ask the computer to count the number of characters and the number of periods in each chapter of both *Huckleberry Finn* and *Little Women*.

In [3]:
# In each chapter, count the number of all characters;
# call this the "length" of the chapter.
# Also count the number of periods.

# POLARS_CHANGE: count_matches treats '.' as a regular expression unless literal=True
huck_finn = pl.Series("chapter", huck_finn_chapters)
chars_periods_huck_finn = pl.DataFrame([
    pl.Series("Huck Finn Chapter Length", [len(s) for s in huck_finn_chapters]),
    huck_finn.str.count_matches(".", literal=True).alias("Number of Periods"),
])
little_women = pl.Series("chapter", little_women_chapters)
chars_periods_little_women = pl.DataFrame([
    pl.Series("Little Women Chapter Length", [len(s) for s in little_women_chapters]),
    little_women.str.count_matches(".", literal=True).alias("Number of Periods"),
])

Here are the data for *Huckleberry Finn*. Each row of the DataFrame corresponds to one chapter of the novel and displays the number of characters as well as the number of periods in the chapter. Not surprisingly, chapters with fewer characters also tend to have fewer periods, in general: the shorter the chapter, the fewer sentences there tend to be, and vice versa. The relation is not entirely predictable, however, as sentences are of varying lengths and can involve other punctuation such as question marks. 

In [4]:
chars_periods_huck_finn

Huck Finn Chapter Length,Number of Periods
i64,u32
7026,66
11982,117
8529,72
6799,84
8166,91
…,…
10864,96
11534,60
13430,77


Here are the corresponding data for *Little Women*.

In [5]:
chars_periods_little_women

Little Women Chapter Length,Number of Periods
i64,u32
21759,189
22148,188
20558,231
25526,195
23395,255
…,…
33202,305
10289,95
12558,96


You can see that the chapters of *Little Women* are in general longer than those of *Huckleberry Finn*. Let us see if these two simple variables – the length and number of periods in each chapter – can tell us anything more about the two books. One way to do this is to plot both sets of data on the same axes. 

In the plot below, there is a dot for each chapter in each book. Blue dots correspond to *Huckleberry Finn* and gold dots to *Little Women*. The horizontal axis represents the number of periods and the vertical axis represents the number of characters.

In [6]:
# POLARS_CHANGE: hvPlot overlays groups from one column, as in visualizations.ipynb
chapter_lengths = pl.concat([
    chars_periods_huck_finn.select(
        pl.col("Number of Periods"),
        pl.col("Huck Finn Chapter Length").alias("Chapter Length"),
        pl.lit("Huckleberry Finn").alias("Book"),
    ),
    chars_periods_little_women.select(
        pl.col("Number of Periods"),
        pl.col("Little Women Chapter Length").alias("Chapter Length"),
        pl.lit("Little Women").alias("Book"),
    ),
])
chapter_lengths.hvplot.scatter(
    x="Number of Periods",
    y="Chapter Length",
    by="Book",
    color=["darkblue", "gold"],
    xlabel="Number of periods in chapter",
    ylabel="Number of characters in chapter",
    width=600,
    height=600,
)

:NdOverlay   [Book]
   :Scatter   [Number of Periods]   (Chapter Length)

The plot shows us that many but not all of the chapters of *Little Women* are longer than those of *Huckleberry Finn*, as we had observed by just looking at the numbers. But it also shows us something more. Notice how the blue points are roughly clustered around a straight line, as are the yellow points. Moreover, it looks as though both colors of points might be clustered around the *same* straight line.

Now look at all the chapters that contain about 100 periods. The plot shows that those chapters contain about 10,000 characters to about 15,000 characters, roughly. That's about 100 to 150 characters per period.

Indeed, it appears from looking at the plot that on average both books tend to have somewhere between 100 and 150 characters between periods, as a very rough estimate. Perhaps these two great 19th century novels were signaling something so very familiar to us now: the 140-character limit of Twitter.